In [2]:
import numpy as np
import tensorflow as tf
from tensorflow import keras

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

np.random.seed(100)
np.random.seed(100)

In [3]:
data = load_breast_cancer()
X = data.data
y = data.target

print(np.shape(X))
print(np.shape(y))

(569, 30)
(569,)


In [4]:
X_trainval, X_test, y_trainval, y_test = train_test_split(
  X, y, test_size=0.2, random_state=100, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
  X_trainval, y_trainval, test_size=0.25, random_state=100, stratify=y_trainval
)

In [5]:
#Normalization
scaler = StandardScaler()

X_train_norm = scaler.fit_transform(X_train)
X_test_norm = scaler.transform(X_test)
X_val_norm = scaler.transform(X_val)

In [6]:
model = keras.Sequential([
  keras.layers.Input(shape=(30,)),
  keras.layers.Dense(1, activation='sigmoid') #Single neuron
  ])

model.compile(
  optimizer = keras.optimizers.Adam(learning_rate = 0.01),
  loss = 'binary_crossentropy',
  metrics = ['accuracy']
  )

In [7]:
history = model.fit(
  X_train_norm, y_train,
  validation_data=(X_val_norm, y_val),
    epochs=100,
    batch_size=32,
    verbose=1
)

Epoch 1/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.8534 - loss: 0.3968 - val_accuracy: 0.9035 - val_loss: 0.2939
Epoch 2/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9413 - loss: 0.2215 - val_accuracy: 0.9123 - val_loss: 0.1968
Epoch 3/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9560 - loss: 0.1629 - val_accuracy: 0.9211 - val_loss: 0.1535
Epoch 4/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9589 - loss: 0.1354 - val_accuracy: 0.9386 - val_loss: 0.1330
Epoch 5/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9707 - loss: 0.1210 - val_accuracy: 0.9386 - val_loss: 0.1224
Epoch 6/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9765 - loss: 0.1124 - val_accuracy: 0.9386 - val_loss: 0.1139
Epoch 7/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9795 - loss: 0.1049 - val_accuracy: 0.9386 - val_loss: 0.1092
Epoch 8/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9795 - loss: 0.0997 - val_accuracy: 0.9386 - 

### Adding Hidden Layers

In [8]:
better_model = keras.Sequential([
  keras.layers.Input(shape=(30,)),
  
  #ReLu hidden layers
  keras.layers.Dense(32, activation='relu'),
  keras.layers.Dense(16, activation='relu'),
  keras.layers.Dense(8, activation='relu'),
  
  #Sigmoid output
  keras.layers.Dense(1, activation='sigmoid')
])

better_model.compile(
  optimizer = keras.optimizers.Adam(0.01),
  loss = 'binary_crossentropy',
  metrics = ['accuracy']
)

better_history = better_model.fit(
  X_train_norm, y_train,
  validation_data = (X_val_norm, y_val),
  epochs = 200, batch_size = 32, verbose = 1
)

Epoch 1/200
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.8328 - loss: 0.3850 - val_accuracy: 0.9211 - val_loss: 0.1565
Epoch 2/200
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9472 - loss: 0.1178 - val_accuracy: 0.9737 - val_loss: 0.0478
Epoch 3/200
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9795 - loss: 0.0737 - val_accuracy: 0.9825 - val_loss: 0.0520
Epoch 4/200
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9853 - loss: 0.0559 - val_accuracy: 0.9737 - val_loss: 0.0620
Epoch 5/200
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9883 - loss: 0.0433 - val_accuracy: 0.9825 - val_loss: 0.0558
Epoch 6/200
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9883 - loss: 0.0384 - val_accuracy: 0.9825 - val_loss: 0.0587
Epoch 7/200
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9912 - loss: 0.0300 - val_accuracy: 0.9737 - val_loss: 0.0572
Epoch 8/200
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9912 - loss: 0.0267 - val_accuracy: 0.9649 - 

### Final Model: 
Including:
- Regularization
- Dropout
- Early stopping

In [9]:
final_model = keras.Sequential([
  keras.layers.Input(shape=(30,)),
  
  #Hidden layers
  keras.layers.Dense(64, activation='relu', kernel_regularizer=keras.regularizers.l2(0.001)),
  keras.layers.Dropout(0.3),

  keras.layers.Dense(32, activation='relu', kernel_regularizer=keras.regularizers.l2(0.001)),
  keras.layers.Dropout(0.2),

  keras.layers.Dense(16, activation='relu'),

  keras.layers.Dense(1, activation='sigmoid')
])

final_model.compile(
  optimizer = keras.optimizers.Adam(learning_rate = 0.001),
  loss = 'binary_crossentropy',
  metrics = ['accuracy']
)

early_stop = keras.callbacks.EarlyStopping(
  monitor = 'val_loss',
  patience = 25,
  restore_best_weights = True
)

final_history = final_model.fit(
  X_train_norm, y_train,
  validation_data = (X_val_norm, y_val),
  epochs = 500,
  batch_size = 32,
  verbose = 1,
  callbacks = [early_stop]
)

Epoch 1/500
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.6569 - loss: 0.6999 - val_accuracy: 0.8596 - val_loss: 0.5962
Epoch 2/500
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8065 - loss: 0.5832 - val_accuracy: 0.9123 - val_loss: 0.4850
Epoch 3/500
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8915 - loss: 0.4626 - val_accuracy: 0.9035 - val_loss: 0.3947
Epoch 4/500
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9208 - loss: 0.3676 - val_accuracy: 0.9123 - val_loss: 0.3252
Epoch 5/500
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9326 - loss: 0.3193 - val_accuracy: 0.9123 - val_loss: 0.2779
Epoch 6/500
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9501 - loss: 0.2870 - val_accuracy: 0.9211 - val_loss: 0.2467
Epoch 7/500
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9472 - loss: 0.2625 - val_accuracy: 0.9386 - val_loss: 0.2191
Epoch 8/500
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9560 - loss: 0.2169 - val_accuracy: 0.9561 - 

In [33]:
test_loss, test_accuracy = final_model.evaluate(X_test_norm, y_test, verbose = 0)
y_pred_probability = final_model.predict(X_test_norm, verbose = 0).flatten()

print("Accuracy: ", round(test_accuracy, 5))
print("Loss    : ", round(test_loss, 5))

Accuracy:  0.99123
Loss    :  0.08849
